# R5 practice: can you trust this battery study?

*Energy Systems, Step by Step* · Researcher tier · practice for studio R5 · about 2 hours.

The page's study asks: **how much would a 50 MW, 200 MWh battery have saved Prairie Town on Wednesday 20 August 2025?** This notebook builds the same one-day linear program in `scipy.optimize.linprog(method="highs")`, with the five planted bugs of the page behind a switch. A saving here is the cut in the day's modeled running cost (the unserved-demand penalty included), not what the battery's owner would earn: its own cost, wear and market prices are left out. You will write each check as a function, turn the benchmark into a test, validate one input against measured data, and rerun the study from a clean clone.

**How to run it.** Google Colab: *File → Upload notebook*, then *Runtime → Run all*. Locally: Python 3.9 or later with `numpy` and `scipy` 1.9 or later. The Git part (section 7) needs a terminal on your own computer. Practice 3 also reads `town_year.json` (download it from the R5 page and put it next to this notebook).

**The data.** Prairie Town is fictional; its plants, costs and battery are illustrative. Demand, wind and solar follow the real hourly shapes of 19 and 20 August and 3 September 2025 on the Southwest Power Pool grid (EIA Form EIA-930, Hourly Electric Grid Monitor), scaled to the town exactly as in lesson 0. The `assert` cells check that you get the browser's numbers.

In [ ]:
import datetime as dt, hashlib, json, math, os
import numpy as np
import scipy
from scipy.optimize import linprog

print("NumPy", np.__version__, "· SciPy", scipy.__version__)
major, minor = (int(v) for v in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this notebook needs SciPy 1.9 or later (pip install -U scipy)"

## 1. The data and the study's inputs

Hours run 0 to 23; each lasts one hour ($\Delta t$ = 1 h), so a power in MW held for one hour is an energy in MWh. Units are written beside every input: a unit you cannot name is a bug waiting to happen.

In [ ]:
# Wednesday 20 August 2025 (the study), Tuesday 19 August (the day before), 3 September (lesson 0's day), MW by hour
demand_study = [310, 301, 288, 281, 282, 283, 295, 307, 321, 334, 354, 376, 389, 404, 418, 425, 425, 421, 415, 405, 388, 373, 347, 325]
wind_study = [45, 36, 33, 36, 32, 33, 35, 32, 25, 16, 15, 16, 18, 23, 26, 26, 27, 30, 31, 32, 41, 53, 57, 60]
solar_study = [0, 0, 0, 0, 0, 0, 0, 15, 71, 108, 154, 170, 176, 166, 147, 149, 148, 151, 104, 43, 2, 0, 0, 0]
demand_before = [326, 308, 299, 295, 292, 297, 305, 317, 326, 347, 368, 387, 409, 424, 433, 440, 439, 432, 420, 405, 394, 377, 351, 331]
wind_before = [36, 33, 24, 20, 17, 17, 15, 16, 14, 12, 9, 6, 6, 9, 12, 18, 29, 45, 47, 47, 61, 67, 56, 51]
solar_before = [0, 0, 0, 0, 0, 0, 0, 16, 92, 140, 154, 156, 153, 155, 156, 157, 130, 122, 93, 53, 3, 0, 0, 0]
demand_lesson0 = [250, 243, 238, 235, 236, 237, 253, 260, 266, 278, 286, 300, 311, 327, 340, 344, 353, 349, 344, 331, 319, 303, 285, 269]
wind_lesson0 = [30, 28, 28, 30, 36, 42, 43, 48, 46, 35, 36, 38, 35, 33, 33, 37, 47, 54, 59, 72, 89, 98, 95, 89]
solar_lesson0 = [0, 0, 0, 0, 0, 0, 0, 7, 70, 118, 130, 147, 156, 149, 112, 147, 161, 130, 70, 14, 0, 0, 0, 0]

DAYS = {
    "2025-08-20": dict(demand=demand_study, wind=wind_study, solar=solar_study),
    "2025-08-19": dict(demand=demand_before, wind=wind_before, solar=solar_before),
    "2025-09-03": dict(demand=demand_lesson0, wind=wind_lesson0, solar=solar_lesson0),
}
# the benchmark: a made-up day whose answer can be worked out by hand
DAYS["benchmark"] = dict(demand=[330 if 12 <= t <= 16 else 220 for t in range(24)], wind=[0] * 24, solar=[0] * 24)

PLANTS = {                         # size (MW), running cost ($/MWh): lesson 0's fleet
    "nuc":   dict(size=100, cost=10),    # runs flat at 100 MW
    "wind":  dict(size=150, cost=0),     # capped by the wind each hour
    "solar": dict(size=200, cost=0),     # capped by the sun each hour
    "cc":    dict(size=150, cost=40),    # gas combined-cycle: changes by at most 40 MW an hour, from 115 MW
    "pk":    dict(size=100, cost=100),   # gas peaker
}
RAMP, CC_START = 40, 115           # MW per hour, MW
V = 10_000                         # $ per MWh of demand left unserved (illustrative)
P = 50                             # battery power, MW
E0 = 100                           # battery energy at the start of the day, MWh
ETA = 0.85                         # round trip, counted on charging (NLR 2025 ATB: 85%)
print({k: max(d["demand"]) for k, d in DAYS.items()}, "MW peak demand")

## 2. The model, with the five planted bugs behind a switch

| Rule | In symbols | Rows |
|---|---|---|
| Goal | $\min \sum_t \big(\sum_g c_g\, p_{g,t} + V u_t\big)$ | 1 |
| Balance | $\sum_g p_{g,t} + d_t - c_t + u_t = D_t$ | 24 |
| Stored energy | $e_t = e_{t-1} + (\eta\, c_t - d_t)\,\Delta t$, with $e_{-1} = E_0$ and $\Delta t = 1$ h | 24 |
| Gas ramp | $-40 \le p_{\text{cc},t} - p_{\text{cc},t-1} \le 40$, with $p_{\text{cc},-1} = 115$ | 24 (two-sided) |
| End of day | $e_{23} = E_0$ | 1 |
| Nuclear plant | $p_{\text{nuc},t} = 100$: it runs flat at its size | bounds |
| Sizes | $0 \le p_{g,t} \le \bar P_g$ for the other plants (wind and solar: the weather), $0 \le c_t, d_t \le 50$, $0 \le e_t \le E$ | bounds |

Symbols: $t$ hour; $g$ plant; $p_{g,t}$ output, MW; $c_t$, $d_t$ battery charging and discharging power, MW; $e_t$ stored energy at the end of hour $t$, MWh; $u_t$ unserved demand, MW; $D_t$ demand, MW; $c_g$ running cost, \$/MWh; $V$ \$/MWh; $\eta = 0.85$, the round trip, all of its loss counted on charging; $E = 200$ MWh; $E_0 = 100$ MWh; $\Delta t = 1$ h. In the code each hour's $\Delta t$ is 1, so the stored-energy row reads $e_t - e_{t-1} - \eta c_t + d_t = 0$.

`bug` is one of `None`, `"unit"` (the energy size typed in kWh), `"sign"` (charging counted as supply), `"lunch"` (no end-of-day rule) or `"missing"` (no ramp rows). The fifth bug, the leaked future, is not in the model: it is which day the battery's schedule is planned on (section 5).

In [ ]:
def build(day, bug=None, eta=ETA, battery=True, fix=None):
    # Returns the linprog inputs, the column of each variable and the name of each row.
    D = DAYS[day]
    E = 200_000 if bug == "unit" else 200          # bug "unit": the spec sheet's 200,000 kWh read as MWh
    names = []
    for t in range(24):
        names += [f"{g}_{t}" for g in PLANTS] + [f"u_{t}"] + ([f"c_{t}", f"d_{t}", f"e_{t}"] if battery else [])
    col = {n: j for j, n in enumerate(names)}
    nv = len(names)
    cost = np.zeros(nv)
    bounds = [(0, None)] * nv
    for t in range(24):
        for g, p in PLANTS.items():
            cost[col[f"{g}_{t}"]] = p["cost"]
            hi = D[g][t] if g in ("wind", "solar") else p["size"]
            bounds[col[f"{g}_{t}"]] = (hi, hi) if g == "nuc" else (0, hi)
        cost[col[f"u_{t}"]] = V
        if battery:
            bounds[col[f"c_{t}"]] = (fix["c"][t],) * 2 if fix else (0, P)
            bounds[col[f"d_{t}"]] = (fix["d"][t],) * 2 if fix else (0, P)
            bounds[col[f"e_{t}"]] = (0, E)
    A_eq, b_eq, eq_names, A_ub, b_ub, ub_names = [], [], [], [], [], []
    def row(terms):
        r = np.zeros(nv)
        for k, v in terms.items():
            r[col[k]] += v
        return r
    for t in range(24):
        terms = {f"{g}_{t}": 1 for g in PLANTS}
        terms[f"u_{t}"] = 1
        if battery:
            s = -1 if bug == "sign" else 1               # bug "sign": the battery's terms flipped in the balance
            terms[f"d_{t}"], terms[f"c_{t}"] = s, -s
        A_eq.append(row(terms)); b_eq.append(D["demand"][t]); eq_names.append(f"bal_{t}")
        if battery:
            terms = {f"e_{t}": 1, f"c_{t}": -eta, f"d_{t}": 1}
            if t:
                terms[f"e_{t-1}"] = -1
            A_eq.append(row(terms)); b_eq.append(0 if t else E0); eq_names.append(f"soc_{t}")
        if bug != "missing":                             # bug "missing": the ramp rows were never written
            prev = {f"cc_{t-1}": -1} if t else {}
            up = row({f"cc_{t}": 1, **prev}); A_ub += [up, -up]
            b_ub += [RAMP if t else CC_START + RAMP, RAMP if t else -(CC_START - RAMP)]
            ub_names += [f"ramp_{t}", f"ramp_{t}"]
    if battery and bug != "lunch":                       # bug "lunch": no end-of-day rule
        A_eq.append(row({"e_23": 1})); b_eq.append(E0); eq_names.append("end")
    return dict(c=cost, A_eq=np.array(A_eq), b_eq=np.array(b_eq), A_ub=np.array(A_ub) if A_ub else None,
                b_ub=np.array(b_ub) if b_ub else None, bounds=bounds, col=col, eq_names=eq_names, ub_names=ub_names,
                day=day, bug=bug, battery=battery)

def solve(day, **kw):
    m = build(day, **kw)
    r = linprog(m["c"], A_ub=m["A_ub"], b_ub=m["b_ub"], A_eq=m["A_eq"], b_eq=m["b_eq"], bounds=m["bounds"], method="highs")
    assert r.status == 0, r.message
    m["cost"], m["x"] = r.fun, {n: r.x[j] for n, j in m["col"].items()}
    m["status"] = "optimal"
    return m

def series(m, k):
    return np.array([m["x"][f"{k}_{t}"] for t in range(24)])

def saving(day="2025-08-20", **kw):
    # what a version's code would report: its own day without a battery minus the same version with it
    base_kw = {k: v for k, v in kw.items() if k == "bug" and v == "missing"}
    return solve(day, battery=False, **base_kw)["cost"] - solve(day, **kw)["cost"]

## 3. The fixed study and the five versions

The cost of a linear program's optimum is unique, so these numbers must match the browser to the cent. The plans need not: when several plans cost the same, another HiGHS version may return another one. Section 5 shows when that matters.

In [ ]:
base = solve("2025-08-20", battery=False)
fixed = solve("2025-08-20")
print(f"Day without a battery: ${base['cost']:,.2f}; with it: ${fixed['cost']:,.2f}; saving ${base['cost'] - fixed['cost']:,.2f}")
assert abs(base["cost"] - 194500) < 0.01
assert abs((base["cost"] - fixed["cost"]) - 8205.88) < 0.01

VERSIONS = {1: "unit", 2: "sign", 3: "lunch", 4: "missing"}
for n, bug in VERSIONS.items():
    print(f"Version {n} ({bug}): reports a saving of ${saving(bug=bug):,.2f}")
assert abs(saving(bug="unit") - 12150) < 0.01
assert abs(saving(bug="lunch") - 18205.88) < 0.01
assert abs(saving(bug="missing") - 8205.88) < 0.01      # the same as the fixed study: the headline does not move

## 4. The checks, as functions

Each check returns a list of problems (empty when it passes). Run them on every version: each bug should fail at least the check the page names, and the fixed study should pass them all. Version 3 also fails the row count (a missing rule is also a missing row): say in your log why two checks catch it.

In [ ]:
TOL = 1e-6

def check_sizes(m):
    # Units and sizes: every result within its physical size (battery 0-50 MW, 0-200 MWh)
    out = []
    if m["battery"]:
        e = series(m, "e")
        if e.max() > 200 + TOL or e.min() < -TOL:
            out.append(f"stored energy from {e.min():.1f} to {e.max():.1f} MWh in a 200 MWh battery")
    return out

def check_balance_signs(m):
    # Balance and signs, recomputed with the physical convention: discharge feeds the town, charge draws from it
    out, D = [], DAYS[m["day"]]
    for t in range(24):
        supply = sum(m["x"][f"{g}_{t}"] for g in PLANTS) + m["x"][f"u_{t}"]
        if m["battery"]:
            supply += m["x"][f"d_{t}"] - m["x"][f"c_{t}"]
        if abs(supply - D["demand"][t]) > 1e-6:
            out.append(f"hour {t}: supply {supply:.1f} MW against demand {D['demand'][t]} MW")
    return out

def check_closure(m):
    # Closure over the day: the battery ends with what it started with
    if m["battery"] and abs(m["x"]["e_23"] - E0) > TOL:
        return [f"stored energy starts at {E0} MWh and ends at {m['x']['e_23']:.1f} MWh"]
    return []

PAPER_ROWS = {"bal": 24, "soc": 24, "ramp": 24, "end": 1}   # from the formulation (each ramp row is two-sided)

def check_paper_rows(m):
    # Paper against code: each rule of the formulation has its rows
    names = m["eq_names"] + m["ub_names"][::2]      # a two-sided ramp row is written as two one-sided rows
    out = []
    for rule, n in PAPER_ROWS.items():
        have = sum(1 for k in names if k.split("_")[0] == rule)
        if m["battery"] or rule in ("bal", "ramp"):
            if have != n:
                out.append(f"{rule}: {have} rows in the code, {n} on paper")
    return out

CHECKS = [check_sizes, check_balance_signs, check_closure, check_paper_rows]
for name, kw in [("fixed", {})] + [(f"version {n}", {"bug": b}) for n, b in VERSIONS.items()]:
    m = solve("2025-08-20", **kw)
    failed = {f.__name__: f(m) for f in CHECKS if f(m)}
    print(f"{name:10s}", "passes every check" if not failed else {k: v[:1] for k, v in failed.items()})

assert not any(f(fixed) for f in CHECKS)
assert check_sizes(solve("2025-08-20", bug="unit"))
assert check_balance_signs(solve("2025-08-20", bug="sign"))
assert check_closure(solve("2025-08-20", bug="lunch"))
assert check_paper_rows(solve("2025-08-20", bug="missing"))

**Version 4 on another day.** On 20 August the missing ramp rows change nothing. On 3 September, lesson 0's day, they do.

In [ ]:
l0 = solve("2025-09-03", battery=False)["cost"]
l0_missing = solve("2025-09-03", battery=False, bug="missing")["cost"]
print(f"3 September without a battery: ${l0:,.0f} with the ramp rows, ${l0_missing:,.0f} without them")
print(f"Battery saving: ${saving('2025-09-03'):,.2f} correct, ${saving('2025-09-03', bug='missing'):,.2f} without the ramp rows")
assert abs(l0 - 104000) < 0.01 and abs(l0_missing - 102600) < 0.01

## 5. The leaked future, and the honest number

The fixed study plans the battery knowing 20 August's demand, wind and solar: perfect foresight. **The decision time:** the schedule is set at midnight, as 20 August begins (local time). By then all 24 hours of 19 August have been measured, and none of 20 August. The simplest forecast is persistence: copy the last complete day. Plan on 19 August's measured hours, then run 20 August with that charge and discharge schedule fixed. Only the battery's schedule is set the day before: the plants are still dispatched knowing all of 20 August, ramp limits included. A fuller test would also dispatch them hour by hour, as each hour is measured.

**A tie rule, fixed in advance.** On 19 August many schedules cost exactly the same: the battery's discharge can move between hours with no change in cost. They do very different things on 20 August, so a saving computed from "whatever plan the solver returned" is not reproducible. The rule, using only 19 August's data: among the least-cost plans, keep the most energy in store (the largest sum of $e_t$). It is a second linear program with the first one's cost as a limit.

In [ ]:
DECISION = dt.datetime(2025, 8, 20, 0, 0)     # the schedule is set at midnight, as 20 August begins (local time)

def check_information(plan_day, decision=DECISION):
    # Information at decision time: every hour the plan uses must have ended by the decision time
    last_hour_ends = dt.datetime.fromisoformat(plan_day) + dt.timedelta(hours=24)
    return [] if last_hour_ends <= decision else [f"plan uses hours up to {last_hour_ends}, after the decision at {decision}"]

def plan_with_rule(day, rule="keep"):
    # Among the least-cost plans for `day`, the one that keeps the most energy in store ("keep": the largest sum of
    # e_t) or empties the battery first ("early": the smallest). Stage 1 finds the least cost; stage 2 keeps the cost
    # within rounding of it (one more row, c . x <= cost) and minimises or maximises the stored energy.
    first = solve(day)
    m = build(day)
    c2 = np.zeros(len(m["c"]))
    for t in range(24):
        c2[m["col"][f"e_{t}"]] = -1 if rule == "keep" else 1
    A_ub = np.vstack([m["A_ub"], m["c"]])
    b_ub = np.append(m["b_ub"], first["cost"] * (1 + 1e-9) + 1e-7)
    r = linprog(c2, A_ub=A_ub, b_ub=b_ub, A_eq=m["A_eq"], b_eq=m["b_eq"], bounds=m["bounds"], method="highs")
    assert r.status == 0, r.message
    x = {n: r.x[j] for n, j in m["col"].items()}
    return dict(c=[x[f"c_{t}"] for t in range(24)], d=[x[f"d_{t}"] for t in range(24)], cost=float(m["c"] @ r.x), first=first["cost"])

keep, early = plan_with_rule("2025-08-19", "keep"), plan_with_rule("2025-08-19", "early")
honest = base["cost"] - solve("2025-08-20", fix=keep)["cost"]
honest_early = base["cost"] - solve("2025-08-20", fix=early)["cost"]
print(f"19 August: least cost ${keep['first']:,.2f}; the rule's plan ${keep['cost']:,.2f}; the other plan ${early['cost']:,.2f} (a tie)")
print(f"Perfect foresight: ${base['cost'] - fixed['cost']:,.2f}; planned the day before with the rule, then followed: ${honest:,.2f}")
print(f"The same day-before cost, but the battery emptied first: ${honest_early:,.2f}")
print("Information check, version 5:", check_information("2025-08-20"), "· fixed:", check_information("2025-08-19"))
assert abs(keep["first"] - 218050) < 0.01 and abs(keep["cost"] - keep["first"]) < 0.01 and abs(early["cost"] - keep["first"]) < 0.01
assert abs(honest - 3600) < 0.01 and abs(honest_early - 300) < 0.01
assert check_information("2025-08-20") and not check_information("2025-08-19")

## 6. Benchmark and validation

**Benchmark.** The made-up day: 220 MW, but 330 MW from 12:00 to 17:00; no wind or solar. Without the battery the nuclear plant gives 100 MW, the gas combined-cycle plant 120 MW in the flat hours (30 MW to spare) and its full 150 MW in the five dear hours, when the peaker adds 80 MW at \$100/MWh. By hand, in three steps:

1. **Discharge.** The battery empties its 200 MWh into the dear hours: 200 MWh × \$100/MWh = \$20,000 of peaker output avoided.
2. **Recharge.** Over the day it puts 200 MWh into store (100 before noon, 100 after, so it ends as it began): 200 ÷ 0.85 = 235.29 MWh of charging from the gas plant's spare room, at \$40/MWh: \$9,411.76.
3. **Saving.** \$20,000 − \$9,411.76 = \$10,588.24.

Make it a test that runs every time.

**Validation of one component.** Compare a prediction with measurements of the real system, with a tolerance set first. The day-before plan's prediction of 20 August is its forecast: 19 August's measured net load (demand minus wind and solar). Tolerance: within the battery's 50 MW in every hour, an illustrative screening choice (a fuller one would also weigh the stored energy and the other plants). This validates the forecast only: the battery's dispatch and both savings stay model results.

**An input check and a sensitivity run (not a validation).** The study assumes an 85% round trip (NLR's 2025 ATB). EIA reports that U.S. utility-scale batteries averaged 82% (monthly round trip, 2019 data; Today in Energy, 12 February 2021). Rerun with 82% to see how much the saving depends on it.

In [ ]:
hand = 200 * (100 - 40 / 0.85)
print(f"Benchmark by hand: ${hand:,.2f}")

def test_benchmark(bug=None):
    return abs(saving("benchmark", bug=bug) - hand) < 0.01

for n, bug in [(0, None)] + list(VERSIONS.items()):
    print(f"{'fixed' if n == 0 else 'version ' + str(n):10s} benchmark saving ${saving('benchmark', bug=bug):,.2f}  matches: {test_benchmark(bug)}")
assert abs(hand - 10588.24) < 0.01 and test_benchmark()
assert not test_benchmark("unit") and not test_benchmark("sign") and not test_benchmark("lunch")
assert test_benchmark("missing")          # a benchmark tests only what it exercises

# Validation: the forecast (19 August's measured net load) against 20 August as measured
TOL_MW = 50                                # set before comparing: the battery's power
net = lambda D: np.array(D["demand"]) - np.array(D["wind"]) - np.array(D["solar"])
miss = net(DAYS["2025-08-19"]) - net(DAYS["2025-08-20"])
worst = int(np.abs(miss).argmax())
print(f"Forecast miss: {np.abs(miss).mean():.1f} MW on average, {np.abs(miss).max():.0f} MW at {worst:02d}:00; "
      f"beyond {TOL_MW} MW in {(np.abs(miss) > TOL_MW).sum()} of 24 hours")
assert abs(np.abs(miss).mean() - 22.1) < 0.05 and np.abs(miss).max() == 55 and worst == 12 and (np.abs(miss) > TOL_MW).sum() == 1
# The tolerance fails, so the claim is narrowed: the day-before saving with this forecast, and perfect foresight as a bound

# Input check and sensitivity run
eta82 = saving(eta=0.82)
print(f"Round trip 85%: ${base['cost'] - fixed['cost']:,.2f}; measured 82%: ${eta82:,.2f}")
assert abs(eta82 - 7939.02) < 0.01

## 7. Practice (about 2 hours)

1. **A check of your own** (20 min). Write `check_cc_ramp(m)`: the gas plant's change from one hour to the next, against 40 MW. Run it on version 4 on 20 August and on 3 September. Why does a check on the *solution* sometimes miss a missing rule that the row count always finds?
2. **The benchmark as a test** (15 min). Move `test_benchmark` and the four checks into a file `checks.py`, and run them from `run.py` before anything is reported. Plant a sixth bug of your own (for example, discharge efficiency applied twice) and see which check catches it.
3. **A robustness check on other days** (25 min). With `town_day()` (the cell below, and `town_year.json` next to this notebook), run the fixed study and the day-before plan on other days of 2025 (in July and August, say). Report the range of savings, not one day. This is a robustness check, not a validation: it shows how far one day's result carries over, using the same model.
4. **Version control and a clean clone** (40 min). In a terminal:

   ```
   mkdir battery-study && cd battery-study
   # save this notebook's model as run.py; like the last cell, it writes results.json
   python run.py && shasum -a 256 results.json > first-run.sha256        # the reference checksum
   python --version > python-version.txt                                   # the Python you used, such as Python 3.12.4
   python -m pip freeze | grep -E "^(numpy|scipy)==" > requirements.txt     # pin the packages (SciPy ships the HiGHS that linprog uses)
   git init && git add run.py checks.py requirements.txt python-version.txt first-run.sha256 && git commit -m "Battery study: model, checks, benchmark"
   git tag v1.0
   cd .. && git clone battery-study fresh && cd fresh && git checkout v1.0
   python --version && cat python-version.txt                              # the same Python? If not, install that version first
   python -m venv .venv && . .venv/bin/activate && pip install -r requirements.txt
   python run.py && shasum -a 256 -c first-run.sha256                    # prints "results.json: OK" if the bytes match
   ```

   The commands are for macOS or Linux (bash or zsh). On Windows, use Git Bash, with `sha256sum` in place of `shasum -a 256` and `.venv/Scripts/activate` in place of `.venv/bin/activate`. The same Python, NumPy and SciPy versions on another operating system usually give the same numbers; a different checksum there is a finding to log, not to hide. If the check fails, find out why before you report anything. K8 (Git and reproducible notebooks) walks through these commands.
5. **Your verification log** (20 min). One page: each check with its result, the benchmark with its hand answer, the validation with its data and tolerance, the input check, the day-before and the perfect-foresight saving with what each assumes, and the tag a clean clone reran with its checksum check. Rate it against the rubric on the page.

In [ ]:
# Practice 3: other days of 2025. town_day() cuts one local day (Central time) out of town_year.json and scales it
# exactly as lesson 0 and the R5 page do. town_year.json's index i is the hour ENDING at start_utc + i hours (UTC).
# The two clock-change days (9 March and 2 November 2025) have 23 or 25 hours and are left out.
try:
    from zoneinfo import ZoneInfo
    CENTRAL = ZoneInfo("America/Chicago")          # needs time-zone data: on Windows, pip install tzdata
except Exception:
    CENTRAL = None                                 # fallback: Central time's 2025 rule, written out

def local_midnight_utc(date):
    d = dt.datetime.fromisoformat(date)
    if CENTRAL is not None:
        return d.replace(tzinfo=CENTRAL).astimezone(dt.timezone.utc)
    daylight = dt.datetime(2025, 3, 9) < d < dt.datetime(2025, 11, 2)   # 2025: daylight time from 9 March to 2 November
    return (d + dt.timedelta(hours=5 if daylight else 6)).replace(tzinfo=dt.timezone.utc)

def js_round(x):
    # the page's rounding (JavaScript's Math.round: halves go up), not Python's round() (halves go to even)
    f = math.floor(x)
    return f + 1 if x - f >= 0.5 else f

def town_day(year, date):
    if date in ("2025-03-09", "2025-11-02"):
        raise ValueError(f"{date} is a clock-change day (23 or 25 hours)")
    t0 = dt.datetime.fromisoformat(year["start_utc"]).replace(tzinfo=dt.timezone.utc)
    i = round((local_midnight_utc(date) - t0).total_seconds() / 3600) + 1     # the hour that begins at local midnight
    cut = lambda a: a[i:i + 24]
    wind = [js_round(v / 1000 * 150) for v in cut(year["wind"])]              # MW: shape x the 150 MW wind farm
    solar = [js_round(v / 1000 * 200) for v in cut(year["solar"])]            # MW: shape x the 200 MW solar farm
    demand = [js_round((v / 1000 * 450 - 100 - wind[t] - solar[t]) / 5) * 5 + 100 + wind[t] + solar[t]   # gas need in 5 MW steps
              for t, v in enumerate(cut(year["demand"]))]
    return dict(demand=demand, wind=wind, solar=solar)

if os.path.exists("town_year.json"):
    year = json.load(open("town_year.json"))
    assert town_day(year, "2025-09-03") == DAYS["2025-09-03"]                 # the same scaling as the page (lesson 0's day)
    assert town_day(year, "2025-08-20") == DAYS["2025-08-20"]
    DAYS["2025-07-15"] = town_day(year, "2025-07-15")                         # add any day like this
    print(f"15 July 2025: perfect-foresight saving ${saving('2025-07-15'):,.2f}")
else:
    print("town_year.json is not next to this notebook: download it from the R5 page for practice 3.")

In [ ]:
# results.json: the reported numbers, rounded to the cent, written to disk; the checksum is of the same bytes, so
# shasum -a 256 results.json gives it too (compare it after a clean-clone rerun)
results = {"saving_perfect_foresight": round(base["cost"] - fixed["cost"], 2), "saving_planned_day_before": round(honest, 2),
           "benchmark_hand": round(hand, 2), "benchmark_model": round(saving("benchmark"), 2), "saving_round_trip_82": round(eta82, 2)}
data = (json.dumps(results, indent=2, sort_keys=True) + "\n").encode("utf-8")
with open("results.json", "wb") as f:
    f.write(data)
print(data.decode())
print("SHA-256 of results.json:", hashlib.sha256(data).hexdigest())
assert open("results.json", "rb").read() == data